# Mini caso 4 — Asistente documental semántico

Este notebook implementa **de principio a fin** el asistente documental semántico sobre el corpus RTVE 23-F descrito en el informe parcial.

A diferencia de un buscador puramente literal, el sistema representa el texto en un espacio vectorial y recupera los documentos o fragmentos más cercanos a la consulta del usuario, devolviendo siempre **evidencia trazable** (`doc_id`, título, enlace al PDF y *score* de similitud).

**Contenido del notebook**

1. Carga de datos y selección de campos.
2. Validación de textos y revisión de longitudes.
3. Fragmentación del corpus en *chunks* con trazabilidad.
4. Representación vectorial: **TF-IDF** (baseline) y **embeddings semánticos** (mejora opcional).
5. Motor de recuperación común (similitud coseno, *top-k*).
6. **Búsqueda por pregunta** del usuario.
7. **Recomendación de documentos relacionados**.
8. Comparación TF-IDF vs embeddings.
9. **Evaluación cuantitativa** mediante *known-item retrieval* (recall@k y MRR).
10. Capa **opcional** de respuesta asistida con LLM (RAG controlado por evidencias).
11. Persistencia de artefactos, riesgos y conclusiones.

El sistema se diseña como un **asistente de recuperación documental**, no como un chatbot generalista. La capa generativa con LLM es una extensión final estrictamente condicionada a usar solo los fragmentos recuperados del corpus.


## 0. Entorno, configuración y reproducibilidad

Se fijan los parámetros del asistente en un único bloque para mantener el notebook reproducible y fácil de ajustar. Todas las rutas son relativas a la raíz del proyecto, de modo que el notebook puede ejecutarse de principio a fin sin editar rutas manualmente.

El baseline (TF-IDF) no requiere conexión a internet. La vía de **embeddings** intenta cargar un modelo multilingüe de `sentence-transformers`; si el modelo no está disponible (por ejemplo, sin conexión), el notebook continúa funcionando solo con el baseline gracias a un control de excepciones.

In [1]:
from pathlib import Path
import re
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

# ----------------------- Reproducibilidad -----------------------
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ----------------------- Configuración del asistente -----------------------
CHUNK_MAX_WORDS = 220     # tamaño máximo de cada fragmento (en palabras)
CHUNK_OVERLAP   = 40      # solape entre fragmentos consecutivos
TOP_K           = 5       # nº de resultados por defecto
SNIPPET_WORDS   = 45      # nº de palabras mostradas como evidencia

# Modelo de embeddings (multilingüe, adecuado para español). Requiere internet la 1ª vez.
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Lista de stopwords en español (sklearn solo trae 'english'); se usa en TF-IDF.
SPANISH_STOPWORDS = [
    "a","al","algo","algunas","algunos","ante","antes","como","con","contra","cual","cuando",
    "de","del","desde","donde","durante","e","el","ella","ellas","ellos","en","entre","era",
    "erais","eran","eras","eres","es","esa","esas","ese","eso","esos","esta","estaba","estaban",
    "estado","estais","estamos","estan","estar","estas","este","esto","estos","estoy","fue",
    "fueron","fui","fuimos","ha","habeis","habia","habian","han","hasta","hay","la","las","le",
    "les","lo","los","mas","me","mi","mis","mucho","muchos","muy","nada","ni","no","nos","nosotros",
    "o","os","otra","otras","otro","otros","para","pero","poco","por","porque","que","quien",
    "quienes","se","sea","sean","segun","ser","si","sido","sin","sobre","sois","somos","son","su",
    "sus","tambien","tanto","te","tenia","tiene","tienen","todo","todos","tu","tus","un","una",
    "unas","uno","unos","vosotros","y","ya","yo","esta","mientras","cada","tras","sus",
]

print("Configuración cargada.")
print(f"  chunk_max_words={CHUNK_MAX_WORDS} | overlap={CHUNK_OVERLAP} | top_k={TOP_K}")
print(f"  modelo embeddings='{EMBED_MODEL_NAME}'")

Configuración cargada.
  chunk_max_words=220 | overlap=40 | top_k=5
  modelo embeddings='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'


## 1. Carga de datos

Se utiliza como base el archivo limpio generado en la fase de limpieza general:

`data/processed/rtve_corpus_clean_base.csv`

Este archivo contiene el texto limpio (`text_clean_base`), metadatos básicos, información institucional y enlaces de trazabilidad.

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "rtve_corpus_clean_base.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset cargado: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Shape: {df.shape}")
df.head()

Dataset cargado: data/processed/rtve_corpus_clean_base.csv
Shape: (167, 25)


,doc_id,source_document_id,title,pages,detail_url,pdf_url,summary,keywords,text_full,text_clean_base,text_length_chars,text_length_words,text_clean_length_chars,text_clean_length_words,moncloa_id,moncloa_section,moncloa_subsection,final_match_status,coverage_moncloa,alpha_ratio,digit_ratio,uppercase_ratio,weird_char_ratio,n_title_years,title_main_year
0,rtve_1860,1860,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,3,https://23fbuscador.rtve.es/document/ocr/1860?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_20_de_febrer...,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones pa...",C/SG/2820/20-02-82 DTOR. Vista oral 2/81,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIENTE AL 20-02-82\n\n-...,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIENTE AL 20-02-82\n\n-...,3934,640,3934,640,moncloa_0099,defensa,cni,high_confidence_match,True,0.777834,0.013726,0.147386,0.000000,1,1982.0
1,rtve_1859,1859,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,4,https://23fbuscador.rtve.es/document/ocr/1859?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_22_de_febrer...,Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el C...,C/SG/2896/22-02-82 Vista oral 2/81 Consejo Supremo de Justicia Militar,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81. del Consejo Supremo de Justicia Militar.\n\n1.- DESARROLLO DE LA SE...,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81. del Consejo Supremo de Justicia Militar.\n\n1.- DESARROLLO DE LA SE...,6417,1018,6417,1018,moncloa_0098,defensa,cni,high_confidence_match,True,0.781985,0.009506,0.195895,0.000156,1,1982.0
2,rtve_1858,1858,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,5,https://23fbuscador.rtve.es/document/ocr/1858?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_24_de_febrer...,Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del Consejo Supremo de Ju...,C/SG/2992/24-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81. del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA...,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81. del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA...,8183,1347,8183,1347,moncloa_0097,defensa,cni,high_confidence_match,True,0.784920,0.011487,0.124085,0.000611,1,1982.0
3,rtve_1857,1857,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,6,https://23fbuscador.rtve.es/document/ocr/1857?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/96_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_25_de_febrer...,El documento recoge el desarrollo de la sesión del Consejo Supremo de Justicia Militar en febrero de 1982 relativa al juicio por los suc...,C/SG/3.081/25-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81 del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA...,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81 del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA...,11151,1826,11151,1826,moncloa_0096,defensa,cni,high_confidence_match,True,0.789257,0.008250,0.128167,0.000538,1,1982.0
4,rtve_1856,1

## 2. Selección de campos útiles

Para un asistente documental no se necesitan todas las columnas del dataset. Los campos mínimos son:

- `doc_id`: identificador trazable del documento;
- `title`: título del documento;
- `text_clean_base`: texto limpio que servirá como base de búsqueda;
- `pdf_url`: enlace al documento original;
- `moncloa_section`: sección institucional, si está disponible;
- `text_clean_length_words`: longitud del texto limpio.

Estos campos permiten recuperar evidencia y mostrar al usuario de dónde procede cada resultado.

In [3]:
required_columns = [
    "doc_id",
    "title",
    "text_clean_base",
    "pdf_url",
    "moncloa_section",
    "text_clean_length_words",
]

missing_columns = [c for c in required_columns if c not in df.columns]
print("Columnas requeridas ausentes:", missing_columns)

# Si faltase la longitud, se recalcula a partir del texto limpio (robustez).
if "text_clean_length_words" in missing_columns and "text_clean_base" in df.columns:
    df["text_clean_length_words"] = df["text_clean_base"].fillna("").str.split().str.len()
    missing_columns = [c for c in required_columns if c not in df.columns]

df_assistant = df[required_columns].copy()
df_assistant["text_clean_base"] = df_assistant["text_clean_base"].fillna("").astype(str)
df_assistant.head()

Columnas requeridas ausentes: []


,doc_id,title,text_clean_base,pdf_url,moncloa_section,text_clean_length_words
0,rtve_1860,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIENTE AL 20-02-82\n\n-...,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_20_de_febrer...,defensa,640
1,rtve_1859,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81. del Consejo Supremo de Justicia Militar.\n\n1.- DESARROLLO DE LA SE...,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_22_de_febrer...,defensa,1018
2,rtve_1858,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81. del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA...,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_24_de_febrer...,defensa,1347
3,rtve_1857,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81 del Consejo Supremo de Justicia Militar.\n\n## 1.- DESARROLLO DE LA...,https://www.rtve.es/contenidos/documentos/23f-desclasificado/96_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_25_de_febrer...,defensa,1826
4,rtve_1856,Vista oral 2/81 del Consejo Supremo de Justicia Militar (26 de febrero de 1982).,"C/SG/3.249/26-02-82\nSG\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral de la causa 2/81, del Consejo Supremo de Justicia Militar.\n\n## 1.-...",https://www.rtve.es/contenidos/documentos/23f-desclasificado/95_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_26_de_febrer...,defensa,1740


## 3. Validación de textos disponibles

Antes de construir el índice de búsqueda se comprueba que todos los documentos tienen texto limpio disponible. El sistema solo es viable si existe suficiente contenido textual para recuperar evidencias relevantes.

In [4]:
text_availability = {
    "n_documents": len(df_assistant),
    "n_text_available": int(df_assistant["text_clean_base"].str.strip().ne("").sum()),
    "n_empty_text": int((df_assistant["text_clean_base"].str.strip() == "").sum()),
    "min_text_length_words": int(df_assistant["text_clean_length_words"].min()),
    "median_text_length_words": float(df_assistant["text_clean_length_words"].median()),
    "mean_text_length_words": round(float(df_assistant["text_clean_length_words"].mean()), 1),
    "max_text_length_words": int(df_assistant["text_clean_length_words"].max()),
}
pd.DataFrame(text_availability.items(), columns=["check", "value"])

,check,value
0,n_documents,167.0
1,n_text_available,167.0
2,n_empty_text,0.0
3,min_text_length_words,72.0
4,median_text_length_words,579.0
5,mean_text_length_words,2075.4
6,max_text_length_words,95293.0


## 4. Revisión de longitudes y necesidad de fragmentación

Un asistente documental puede trabajar a nivel de documento completo o a nivel de fragmento. El EDA general mostró que el corpus es muy heterogéneo: conviven documentos breves con documentos muy extensos. Para preguntas concretas, trabajar con documentos completos puede ser poco preciso, porque un texto largo mezcla muchos temas. Se revisa cuántos documentos superan umbrales razonables de longitud.

In [5]:
length_thresholds = [1000, 3000, 5000, 10000]
length_review = []
for threshold in length_thresholds:
    n_docs = int((df_assistant["text_clean_length_words"] > threshold).sum())
    length_review.append({
        "threshold_words": threshold,
        "n_documents_above_threshold": n_docs,
        "percentage": round(n_docs / len(df_assistant) * 100, 2),
    })
pd.DataFrame(length_review)

,threshold_words,n_documents_above_threshold,percentage
0,1000,63,37.72
1,3000,18,10.78
2,5000,9,5.39
3,10000,6,3.59


In [6]:
(df_assistant
    .sort_values("text_clean_length_words", ascending=False)
    .head(10)[["doc_id", "title", "moncloa_section", "text_clean_length_words", "pdf_url"]])

,doc_id,title,moncloa_section,text_clean_length_words,pdf_url
161,rtve_1699,Transcripción de cintas grabadas con conversaciones telefónicas con varias personas intervenidas a la esposa de Tejero.,interior,95293,https://www.rtve.es/contenidos/documentos/23f-desclasificado/07_2026_transcripcion_de_cintas_grabadas_con_conversaciones_telefonicas_con...
159,rtve_1701,Télex interiores y de agencias recibidos en 2ª sección EM el día 23-F informando del estado de situación (23 de febrero de 1981).,interior,19857,https://www.rtve.es/contenidos/documentos/23f-desclasificado/09_1981_telex_interiores_y_de_agencias_recibidos_en_2%C2%AA_seccion_em_el_d...
63,rtve_1797,Investigación y declaraciones personal AOME por JDDI (9 de abril de 1981).,defensa,14485,https://www.rtve.es/contenidos/documentos/23f-desclasificado/36_1981_investigacion_y_declaraciones_personal_aome_por_jddi_9_de_abril_de_...
76,rtve_1784,Policía Nacional. Informe de situación. Marca: reservado-confidencial (12 de noviembre de 1981).,interior,13639,https://www.rtve.es/contenidos/documentos/23f-desclasificado/23_1981_policia_nacional_informe_de_situacion_marca_reservado_confidencial_...
74,rtve_1786,"""Juicio del 23-F: acotaciones al desarrollo del juicio",interior,11080,https://www.rtve.es/contenidos/documentos/23f-desclasificado/25_2026_juicio_del_23_f_acotaciones_al_desarrollo_del_juicio_notas_procesal...
126,rtve_1734,"""Nota Informativa sobre la repercusión en prensa del arresto de Tejero en 1978 cuando era Jefe de la Comandancia de Málaga",interior,10253,https://www.rtve.es/contenidos/documentos/23f-desclasificado/12_1978_nota_informativa_sobre_la_repercusion_en_prensa_del_arresto_de_teje...
50,rtve_1810,Relato de los sucesos de los días 23 y 24 de febrero.,defensa,8965,https://www.rtve.es/contenidos/documentos/23f-desclasificado/49_2026_relato_de_los_sucesos_de_los_dias_23_y_24_de_febrero.pdf
157,rtve_1703,Semestral de la amenaza interior (10 de febrero de 1981; 9 de marzo de 1981).,defensa,8714,https://www.rtve.es/contenidos/documentos/23f-desclasificado/101_1981_semestral_de_la_amenaza_interior_10_de_febrero_de_1981_9_de_marzo_...
93,rtve_1767,"""Informe de las distintas Jefaturas Superiores: Comisaría General de Información. """"Situación actual en las distintas regiones policiale...",interior,7189,https://www.rtve.es/contenidos/documentos/23f-desclasificado/15_1981_informe_de_las_distintas_jefaturas_superiores_comisaria_general_de_...
163,rtve_1697,"""Documentación con una presunta planificación del golpe",interior,4882,https://www.rtve.es/contenidos/documentos/23f-desclasificado/05_1980_documentacion_con_una_presunta_planificacion_del_golpe_manuscrita_1...


**Lectura del bloque.** La revisión confirma que la fragmentación es necesaria para una parte del corpus: una búsqueda a nivel de documento completo recuperaría el documento correcto, pero no necesariamente el pasaje que responde a la pregunta. Por ello, el sistema indexa **fragmentos** (`chunks`) manteniendo la trazabilidad al documento de origen mediante `doc_id`, título y enlace al PDF.

## 5. Fragmentación del corpus en *chunks*

Se divide cada documento en fragmentos solapados de longitud acotada (`CHUNK_MAX_WORDS`), con un solape (`CHUNK_OVERLAP`) que evita cortar ideas justo en el límite entre fragmentos.

- Los documentos cortos (≤ `CHUNK_MAX_WORDS`) generan un único fragmento.
- Los documentos largos se reparten en varios fragmentos numerados.
- Cada fragmento conserva su `doc_id`, título y enlace al PDF, de modo que cualquier resultado de búsqueda puede rastrearse hasta el documento original.

In [7]:
def chunk_document(text, max_words=CHUNK_MAX_WORDS, overlap=CHUNK_OVERLAP):
    """Divide un texto en fragmentos solapados a nivel de palabra."""
    words = text.split()
    if len(words) <= max_words:
        return [{"chunk_index": 0, "n_words": len(words), "text": " ".join(words)}]
    chunks, step, start, idx = [], max_words - overlap, 0, 0
    while start < len(words):
        piece = words[start:start + max_words]
        if not piece:
            break
        chunks.append({"chunk_index": idx, "n_words": len(piece), "text": " ".join(piece)})
        idx += 1
        start += step
    return chunks


records = []
for _, row in df_assistant.iterrows():
    for ch in chunk_document(row["text_clean_base"]):
        records.append({
            "chunk_id": f"{row['doc_id']}::c{ch['chunk_index']:03d}",
            "doc_id": row["doc_id"],
            "title": row["title"],
            "pdf_url": row["pdf_url"],
            "moncloa_section": row["moncloa_section"],
            "chunk_index": ch["chunk_index"],
            "n_words": ch["n_words"],
            "text": ch["text"],
        })

df_chunks = pd.DataFrame(records).reset_index(drop=True)

print(f"Documentos: {df_assistant['doc_id'].nunique()}")
print(f"Fragmentos totales: {len(df_chunks)}")
print(f"Media de fragmentos por documento: {len(df_chunks)/df_assistant['doc_id'].nunique():.2f}")
print(f"Máx. fragmentos en un documento: {df_chunks['doc_id'].value_counts().max()}")
df_chunks["n_words"].describe().round(1)

Documentos: 167
Fragmentos totales: 1998
Media de fragmentos por documento: 11.96
Máx. fragmentos en un documento: 530


count    1998.0
mean      209.9
std        35.7
min         2.0
25%       220.0
50%       220.0
75%       220.0
max       220.0
Name: n_words, dtype: float64

In [8]:
# Distribución de fragmentos por documento (top-10 documentos más fragmentados)
(df_chunks["doc_id"].value_counts()
    .head(10)
    .rename_axis("doc_id")
    .reset_index(name="n_chunks"))

,doc_id,n_chunks
0,rtve_1699,530
1,rtve_1701,111
2,rtve_1797,81
3,rtve_1784,76
4,rtve_1786,62
5,rtve_1734,57
6,rtve_1810,50
7,rtve_1703,49
8,rtve_1767,40
9,rtve_1697,28


## 6. Representación vectorial — baseline TF-IDF

El baseline representa cada fragmento mediante **TF-IDF** con unigramas y bigramas. Es interpretable, rápido y no requiere modelos externos. El vectorizador aplica:

- *stopwords* en español y minúsculas;
- unigramas y bigramas (`ngram_range=(1, 2)`) para captar expresiones como *guardia civil* o *coronel tejero*;
- `min_df=2` para descartar ruido OCR muy infrecuente;
- normalización L2 (por defecto), de modo que el producto escalar equivale a la **similitud coseno**.

In [9]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=SPANISH_STOPWORDS,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    sublinear_tf=True,
)

tfidf_chunk_matrix = tfidf_vectorizer.fit_transform(df_chunks["text"])

print(f"Matriz TF-IDF de fragmentos: {tfidf_chunk_matrix.shape}")
print(f"Tamaño del vocabulario: {len(tfidf_vectorizer.vocabulary_)}")

Matriz TF-IDF de fragmentos: (1998, 52850)
Tamaño del vocabulario: 52850


## 7. Representación semántica — embeddings (mejora opcional)

Como mejora sobre el baseline se generan **embeddings** de los fragmentos con un modelo multilingüe de `sentence-transformers`, capaz de captar similitud de significado aunque no coincidan las palabras exactas.

El bloque está protegido con un control de excepciones: si el modelo no puede descargarse o cargarse (por ejemplo, sin conexión a internet), se activa la bandera `EMBEDDINGS_AVAILABLE = False` y el resto del notebook funciona con el baseline TF-IDF. Esto mantiene el notebook **siempre ejecutable**.

In [10]:
EMBEDDINGS_AVAILABLE = False
embed_model = None
chunk_embeddings = None

try:
    from sentence_transformers import SentenceTransformer
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    chunk_embeddings = embed_model.encode(
        df_chunks["text"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,   # permite usar coseno como producto escalar
    )
    chunk_embeddings = np.asarray(chunk_embeddings)
    EMBEDDINGS_AVAILABLE = True
    print(f"Embeddings generados: {chunk_embeddings.shape}")
except Exception as exc:
    print("[Aviso] Embeddings no disponibles en este entorno; se usará solo TF-IDF.")
    print(f"        Motivo: {type(exc).__name__}: {exc}")

print(f"EMBEDDINGS_AVAILABLE = {EMBEDDINGS_AVAILABLE}")

[Aviso] Embeddings no disponibles en este entorno; se usará solo TF-IDF.
        Motivo: OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
EMBEDDINGS_AVAILABLE = False


## 8. Motor de recuperación

Ambas representaciones comparten la misma lógica: se vectoriza la consulta, se calcula la **similitud coseno** frente a todos los fragmentos y se devuelven los más cercanos. La función `buscar` admite:

- `metodo`: `"tfidf"` o `"embeddings"`;
- `nivel`: `"documento"` (mejor fragmento por documento, evita duplicados del mismo PDF) o `"fragmento"` (devuelve los fragmentos sin agrupar);
- `seccion`: filtro opcional por `moncloa_section`.

Cada resultado incluye `score`, `doc_id`, título, sección, índice de fragmento, un fragmento de evidencia y el enlace al PDF.

In [11]:
def _similitudes(query, metodo):
    """Devuelve el vector de similitudes coseno consulta-fragmentos."""
    if metodo == "tfidf":
        q = tfidf_vectorizer.transform([query])
        return cosine_similarity(q, tfidf_chunk_matrix).ravel()
    if metodo == "embeddings":
        if not EMBEDDINGS_AVAILABLE:
            raise RuntimeError("Embeddings no disponibles: usa metodo='tfidf'.")
        q = embed_model.encode([query], normalize_embeddings=True)
        return cosine_similarity(np.asarray(q), chunk_embeddings).ravel()
    raise ValueError("metodo debe ser 'tfidf' o 'embeddings'")


def _snippet(text, n=SNIPPET_WORDS):
    words = text.split()
    s = " ".join(words[:n])
    return s + (" ..." if len(words) > n else "")


def buscar(query, k=TOP_K, metodo="tfidf", nivel="documento", seccion=None):
    """Búsqueda semántica sobre el corpus. Devuelve evidencias trazables."""
    sims = _similitudes(query, metodo)
    res = df_chunks[["chunk_id", "doc_id", "title", "moncloa_section",
                     "chunk_index", "pdf_url", "text"]].copy()
    res["score"] = sims

    if seccion is not None:
        res = res[res["moncloa_section"] == seccion]

    res = res.sort_values("score", ascending=False)
    if nivel == "documento":
        res = res.drop_duplicates("doc_id")     # mejor fragmento por documento

    res = res.head(k).reset_index(drop=True)
    res["evidencia"] = res["text"].apply(_snippet)
    res["score"] = res["score"].round(4)
    return res[["score", "doc_id", "title", "moncloa_section",
                "chunk_index", "evidencia", "pdf_url"]]


def mostrar_resultados(query, k=TOP_K, metodo="tfidf", nivel="documento", seccion=None):
    """Imprime los resultados en un formato legible de 'asistente'."""
    print(f"Consulta: {query!r}  | método={metodo} | nivel={nivel}"
          + (f" | sección={seccion}" if seccion else ""))
    print("=" * 100)
    out = buscar(query, k=k, metodo=metodo, nivel=nivel, seccion=seccion)
    for i, r in out.iterrows():
        print(f"[{i+1}] score={r['score']:.4f}  doc_id={r['doc_id']}  "
              f"sección={r['moncloa_section']}  (fragmento #{r['chunk_index']})")
        print(f"    Título : {r['title']}")
        print(f"    Evidencia: {r['evidencia']}")
        print(f"    PDF    : {r['pdf_url']}")
        print("-" * 100)
    return out

## 9. Búsqueda por pregunta del usuario

Se definen consultas de ejemplo coherentes con el dominio del 23-F y se ejecuta el asistente. Cada respuesta muestra los documentos más relevantes con su evidencia y enlace al PDF.

In [12]:
example_queries = [
    "¿Qué documentos mencionan al CESID?",
    "¿Qué aparece sobre Tejero en las vistas orales?",
    "¿Qué documentos hablan de la Guardia Civil?",
    "¿Qué documentos tratan sobre la sentencia?",
    "¿Qué actores institucionales aparecen en documentos de Defensa?",
]
pd.DataFrame({"example_query": example_queries})

,example_query
0,¿Qué documentos mencionan al CESID?
1,¿Qué aparece sobre Tejero en las vistas orales?
2,¿Qué documentos hablan de la Guardia Civil?
3,¿Qué documentos tratan sobre la sentencia?
4,¿Qué actores institucionales aparecen en documentos de Defensa?


In [13]:
# Demostración detallada con una consulta (baseline TF-IDF, nivel documento)
_ = mostrar_resultados(example_queries[1], k=5, metodo="tfidf", nivel="documento")

Consulta: '¿Qué aparece sobre Tejero en las vistas orales?'  | método=tfidf | nivel=documento
[1] score=0.0856  doc_id=rtve_1699  sección=interior  (fragmento #229)
    Título : Transcripción de cintas grabadas con conversaciones telefónicas con varias personas intervenidas a la esposa de Tejero.
    Evidencia: --- FIN DE LA CARA 1 DE ESTA CINTA CONVERSACIONES DE LA ESPOSA DEL TTE. CORONEL TEJERO CON OTRAS PERSONAS (2a parte de la cinta) A - Diga? B - ¿Carmen? A - ¿Quien es? B - Herminio. A - Herminio ¿ha visto hijo ... ...
    PDF    : https://www.rtve.es/contenidos/documentos/23f-desclasificado/07_2026_transcripcion_de_cintas_grabadas_con_conversaciones_telefonicas_con_varias_perso.pdf
----------------------------------------------------------------------------------------------------
[2] score=0.0713  doc_id=rtve_1784  sección=interior  (fragmento #27)
    Título : Policía Nacional. Informe de situación. Marca: reservado-confidencial (12 de noviembre de 1981).
    Evidencia: CEHE, C

In [14]:
# Tabla resumen: top-3 documentos para cada consulta de ejemplo (TF-IDF)
filas = []
for q in example_queries:
    top = buscar(q, k=3, metodo="tfidf", nivel="documento")
    for rank, r in top.iterrows():
        filas.append({
            "consulta": q,
            "rank": rank + 1,
            "score": r["score"],
            "doc_id": r["doc_id"],
            "seccion": r["moncloa_section"],
            "title": r["title"][:70],
        })
pd.DataFrame(filas)

,consulta,rank,score,doc_id,seccion,title
0,¿Qué documentos mencionan al CESID?,1,0.1243,rtve_1860,defensa,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero
1,¿Qué documentos mencionan al CESID?,2,0.1007,rtve_1743,exteriores,D.1._AGMAE_R39017_Exp._4
2,¿Qué documentos mencionan al CESID?,3,0.0919,rtve_1698,interior,Documento manuscrito de posible planificación del golpe.
3,¿Qué aparece sobre Tejero en las vistas orales?,1,0.0856,rtve_1699,interior,Transcripción de cintas grabadas con conversaciones telefónicas con va
4,¿Qué aparece sobre Tejero en las vistas orales?,2,0.0713,rtve_1784,interior,Policía Nacional. Informe de situación. Marca: reservado-confidencial
5,¿Qué aparece sobre Tejero en las vistas orales?,3,0.0706,rtve_1767,interior,"""Informe de las distintas Jefaturas Superiores: Comisaría General de I"
6,¿Qué documentos hablan de la Guardia Civil?,1,0.0807,rtve_1743,exteriores,D.1._AGMAE_R39017_Exp._4
7,¿Qué documentos hablan de la Guardia Civil?,2,0.0736,rtve_1698,interior,Documento manuscrito de posible planificación del golpe.
8,¿Qué documentos hablan de la Guardia Civil?,3,0.0681,rtve_1696,interior,Conversaciones telefónicas de (presuntamente) la unidad militar El Par
9,¿Qué documentos tratan sobre la sentencia?,1,0.0970,rtve_1743,exteriores,D.1._AGMAE_R39017_Exp._4


## 10. Recomendación de documentos relacionados

La segunda funcionalidad recomienda documentos similares a uno dado. Se trabaja a **nivel de documento completo**: para TF-IDF se vectoriza el texto limpio de cada documento con el vocabulario ya ajustado; para embeddings se promedian los vectores de sus fragmentos. La recomendación es el conjunto de documentos con mayor similitud coseno al documento de referencia (excluyéndolo a él mismo).

In [15]:
# Representaciones a nivel de documento
doc_ids = df_assistant["doc_id"].tolist()
doc_index = {d: i for i, d in enumerate(doc_ids)}

# TF-IDF de documento completo (mismo vocabulario que los fragmentos)
doc_tfidf_matrix = tfidf_vectorizer.transform(df_assistant["text_clean_base"])

# Embeddings de documento = media de los embeddings de sus fragmentos
doc_embeddings = None
if EMBEDDINGS_AVAILABLE:
    tmp = (pd.DataFrame(chunk_embeddings, index=df_chunks["doc_id"])
             .groupby(level=0).mean())
    tmp = tmp.loc[doc_ids]                       # mismo orden que doc_ids
    doc_embeddings = tmp.to_numpy()
    norms = np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    doc_embeddings = doc_embeddings / np.clip(norms, 1e-12, None)

meta = df_assistant.set_index("doc_id")


def recomendar_documentos(doc_id, k=TOP_K, metodo="tfidf"):
    if doc_id not in doc_index:
        raise KeyError(f"doc_id desconocido: {doc_id}")
    i = doc_index[doc_id]
    if metodo == "embeddings" and EMBEDDINGS_AVAILABLE:
        sims = cosine_similarity(doc_embeddings[i:i+1], doc_embeddings).ravel()
    else:
        sims = cosine_similarity(doc_tfidf_matrix[i], doc_tfidf_matrix).ravel()
    order = np.argsort(-sims)
    order = [j for j in order if j != i][:k]
    return pd.DataFrame({
        "score": np.round(sims[order], 4),
        "doc_id": [doc_ids[j] for j in order],
        "title": [meta.loc[doc_ids[j], "title"] for j in order],
        "moncloa_section": [meta.loc[doc_ids[j], "moncloa_section"] for j in order],
        "pdf_url": [meta.loc[doc_ids[j], "pdf_url"] for j in order],
    })


ejemplo_doc = doc_ids[0]
print(f"Documento de referencia: {ejemplo_doc} — {meta.loc[ejemplo_doc, 'title']}")
recomendar_documentos(ejemplo_doc, k=5, metodo="tfidf")

Documento de referencia: rtve_1860 — Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).


,score,doc_id,title,moncloa_section,pdf_url
0,0.1796,rtve_1702,Vista oral 2/81 del Consejo Supremo de Justicia Militar (19 de febrero de 1982).,defensa,https://www.rtve.es/contenidos/documentos/23f-desclasificado/100_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_19_de_febre...
1,0.1754,rtve_1859,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,defensa,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_22_de_febrer...
2,0.1506,rtve_1857,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,defensa,https://www.rtve.es/contenidos/documentos/23f-desclasificado/96_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_25_de_febrer...
3,0.1491,rtve_1858,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,defensa,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_24_de_febrer...
4,0.1421,rtve_1810,Relato de los sucesos de los días 23 y 24 de febrero.,defensa,https://www.rtve.es/contenidos/documentos/23f-desclasificado/49_2026_relato_de_los_sucesos_de_los_dias_23_y_24_de_febrero.pdf


## 11. Comparación TF-IDF vs embeddings

Cuando los embeddings están disponibles, se compara el solapamiento entre los *top-k* documentos recuperados por cada método para las consultas de ejemplo. Un solapamiento parcial es esperable: TF-IDF prioriza la coincidencia léxica, mientras que los embeddings capturan similitud semántica.

In [16]:
def comparar_metodos(queries, k=5):
    filas = []
    for q in queries:
        d_tfidf = set(buscar(q, k=k, metodo="tfidf")["doc_id"])
        if EMBEDDINGS_AVAILABLE:
            d_emb = set(buscar(q, k=k, metodo="embeddings")["doc_id"])
            inter = len(d_tfidf & d_emb)
            jacc = inter / len(d_tfidf | d_emb) if (d_tfidf | d_emb) else 0.0
            filas.append({"consulta": q, "solapan_topk": inter,
                          "jaccard_topk": round(jacc, 3)})
        else:
            filas.append({"consulta": q, "solapan_topk": np.nan, "jaccard_topk": np.nan})
    return pd.DataFrame(filas)

if EMBEDDINGS_AVAILABLE:
    display(comparar_metodos(example_queries, k=5))
else:
    print("Embeddings no disponibles en este entorno: comparación omitida.")
    print("El notebook funciona íntegramente con el baseline TF-IDF.")

Embeddings no disponibles en este entorno: comparación omitida.
El notebook funciona íntegramente con el baseline TF-IDF.


## 12. Evaluación cuantitativa — *known-item retrieval*

El corpus no dispone de un conjunto de preguntas con respuestas etiquetadas (lo señala el informe), por lo que no es posible una evaluación de relevancia clásica. Para obtener una métrica **objetiva y reproducible** se aplica una evaluación de *known-item retrieval*:

1. Se toma una muestra de documentos.
2. De cada documento se extrae un fragmento interno como **pseudo-consulta**.
3. Se comprueba si el sistema recupera el documento de origen y en qué posición.

Se reportan **recall@1, recall@3, recall@5** y **MRR** (Mean Reciprocal Rank) a nivel de documento. Es una prueba de **consistencia y capacidad de recuperación** del índice; no sustituye la evaluación de relevancia frente a un usuario, pero sí valida que la representación y la similitud funcionan correctamente.

In [17]:
def evaluar_known_item(metodo="tfidf", n_muestra=60, k_max=5, snippet_words=25, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    # documentos suficientemente largos para extraer una pseudo-consulta representativa
    candidatos = df_assistant[df_assistant["text_clean_length_words"] >= 60]["doc_id"].tolist()
    n_muestra = min(n_muestra, len(candidatos))
    muestra = rng.choice(candidatos, size=n_muestra, replace=False)

    recall_at = {1: 0, 3: 0, 5: 0}
    reciprocal_ranks = []
    for d in muestra:
        words = meta.loc[d, "text_clean_base"].split()
        mid = max(0, len(words) // 2 - snippet_words // 2)
        pseudo_query = " ".join(words[mid:mid + snippet_words])
        top = buscar(pseudo_query, k=k_max, metodo=metodo, nivel="documento")
        ids = top["doc_id"].tolist()
        rank = ids.index(d) + 1 if d in ids else None
        for kk in recall_at:
            if rank is not None and rank <= kk:
                recall_at[kk] += 1
        reciprocal_ranks.append(1.0 / rank if rank else 0.0)

    return {
        "metodo": metodo,
        "n_consultas": int(n_muestra),
        "recall@1": round(recall_at[1] / n_muestra, 3),
        "recall@3": round(recall_at[3] / n_muestra, 3),
        "recall@5": round(recall_at[5] / n_muestra, 3),
        "MRR@5": round(float(np.mean(reciprocal_ranks)), 3),
    }


resultados_eval = [evaluar_known_item(metodo="tfidf", n_muestra=60)]
if EMBEDDINGS_AVAILABLE:
    resultados_eval.append(evaluar_known_item(metodo="embeddings", n_muestra=60))

df_eval = pd.DataFrame(resultados_eval)
df_eval

,metodo,n_consultas,recall@1,recall@3,recall@5,MRR@5
0,tfidf,60,1.0,1.0,1.0,1.0


In [18]:
# Figura: recall@k por método (se guarda en outputs/figures)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig_dir = PROJECT_ROOT / "outputs" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

ks = ["recall@1", "recall@3", "recall@5"]
fig, ax = plt.subplots(figsize=(7, 4))
width = 0.35
x = np.arange(len(ks))
for i, row in df_eval.iterrows():
    ax.bar(x + i * width, [row[k] for k in ks], width=width, label=row["metodo"])
ax.set_xticks(x + width * (len(df_eval) - 1) / 2)
ax.set_xticklabels(ks)
ax.set_ylim(0, 1.05)
ax.set_ylabel("recall")
ax.set_title("Mini caso 4 — Known-item retrieval (recall@k)")
ax.legend()
fig.tight_layout()
fig_path = fig_dir / "caso4_recall_at_k.png"
fig.savefig(fig_path, dpi=120)
plt.close(fig)
print(f"Figura guardada en: {fig_path.relative_to(PROJECT_ROOT)}")

Figura guardada en: outputs/figures/caso4_recall_at_k.png


## 13. Capa opcional — respuesta asistida con LLM (RAG controlado)

Esta capa es una **extensión final opcional**. Convierte el asistente de recuperación en un sistema tipo RAG: recupera fragmentos del corpus y pide a un LLM que redacte una respuesta **basada exclusivamente en esas evidencias**, citando los `doc_id` usados y reconociendo cuando la información no aparece en el corpus.

El bloque está protegido: solo se ejecuta si la librería `anthropic` está instalada y existe la variable de entorno `ANTHROPIC_API_KEY`. En caso contrario, el notebook muestra el *prompt* y el contexto que se enviarían, sin llamar a ningún servicio externo. Así se respeta el principio del informe: **toda respuesta debe apoyarse en evidencias trazables del corpus**.

In [19]:
import os

SYSTEM_PROMPT = (
    "Eres un asistente documental sobre los documentos desclasificados del 23-F. "
    "Responde ÚNICAMENTE con la información contenida en los fragmentos proporcionados. "
    "No añadas datos externos ni conocimiento general. "
    "Cita siempre los doc_id en los que te apoyas. "
    "Si los fragmentos no contienen la respuesta, dilo explícitamente: "
    "'No hay evidencia suficiente en el corpus para responder.'"
)


def construir_contexto(query, k=5, metodo=None):
    """Recupera evidencias y arma el contexto para el LLM."""
    metodo = metodo or ("embeddings" if EMBEDDINGS_AVAILABLE else "tfidf")
    ev = buscar(query, k=k, metodo=metodo, nivel="documento")
    bloques = []
    for _, r in ev.iterrows():
        bloques.append(f"[doc_id={r['doc_id']} | {r['title']}]\n{r['evidencia']}")
    contexto = "\n\n".join(bloques)
    return ev, contexto


def responder_con_evidencia(query, k=5, metodo=None, model="claude-3-5-sonnet-latest"):
    ev, contexto = construir_contexto(query, k=k, metodo=metodo)
    user_msg = (f"Pregunta: {query}\n\n"
                f"Fragmentos recuperados del corpus:\n{contexto}\n\n"
                f"Responde apoyándote solo en estos fragmentos y cita los doc_id.")

    if "ANTHROPIC_API_KEY" not in os.environ:
        print("[Modo sin LLM] No hay ANTHROPIC_API_KEY; se muestra el prompt y el contexto.")
        print("--- SYSTEM ---\n" + SYSTEM_PROMPT)
        print("--- USER ---\n" + user_msg)
        return {"answer": None, "evidencia": ev}
    try:
        import anthropic
        client = anthropic.Anthropic()
        resp = client.messages.create(
            model=model, max_tokens=600, system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
        )
        answer = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
        print(answer)
        return {"answer": answer, "evidencia": ev}
    except Exception as exc:
        print(f"[Aviso] No se pudo llamar al LLM: {type(exc).__name__}: {exc}")
        return {"answer": None, "evidencia": ev}


# Demostración (en modo sin LLM solo imprime el prompt y devuelve la evidencia)
_demo = responder_con_evidencia(example_queries[0], k=3)
_demo["evidencia"]

[Modo sin LLM] No hay ANTHROPIC_API_KEY; se muestra el prompt y el contexto.
--- SYSTEM ---
Eres un asistente documental sobre los documentos desclasificados del 23-F. Responde ÚNICAMENTE con la información contenida en los fragmentos proporcionados. No añadas datos externos ni conocimiento general. Cita siempre los doc_id en los que te apoyas. Si los fragmentos no contienen la respuesta, dilo explícitamente: 'No hay evidencia suficiente en el corpus para responder.'
--- USER ---
Pregunta: ¿Qué documentos mencionan al CESID?

Fragmentos recuperados del corpus:
[doc_id=rtve_1860 | Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).]
REY y paralelamente el de la REINA; se afirma tenían conocimiento de lo que se proponían realizar los procesados. La lectura de declaraciones de algunos procesados hace que el nombre de Su Majestad esté presente de modo continuado. En quienes no tengan un juicio formado ...

[doc_id=rtve_1743 | D.1._AGMAE_R39017_Exp._4]
JLBB/ng M

,score,doc_id,title,moncloa_section,chunk_index,evidencia,pdf_url
0,0.1243,rtve_1860,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,defensa,2,REY y paralelamente el de la REINA; se afirma tenían conocimiento de lo que se proponían realizar los procesados. La lectura de declarac...,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_militar_20_de_febrer...
1,0.1007,rtve_1743,D.1._AGMAE_R39017_Exp._4,exteriores,0,"JLBB/ng MINISTERIO DE ASUNTOS EXTERIORES DIRECCION GENERAL DE POLITICA EXTERIOR PARA AMERICA DEL NORTE Y PACIFICO Madrid, 13 de marzo de...",https://www.rtve.es/contenidos/documentos/23f-desclasificado/138_2026_d1_agmae_r39017_exp_4.pdf
2,0.0919,rtve_1698,Documento manuscrito de posible planificación del golpe.,interior,1,"de UME. [fecha] 44 - de generales con fecha probable R.A. - de generales del C.G. - de defiantes militares con ""apoyos"". - a máquina s'd...",https://www.rtve.es/contenidos/documentos/23f-desclasificado/06_2026_documento_manuscrito_de_posible_planificacion_del_golpe.pdf


## 14. Persistencia de artefactos

Se guardan los artefactos reutilizables por otros mini casos y por el informe, siempre con rutas relativas a la raíz del proyecto:

- la tabla de fragmentos (`data/processed/rtve_corpus_chunks.csv`);
- ejemplos de búsqueda (`outputs/tables/caso4_busqueda_ejemplos.csv`);
- el índice TF-IDF serializado (`outputs/models/caso4_tfidf_index.joblib`).

In [20]:
import joblib

(PROJECT_ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "outputs" / "tables").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "outputs" / "models").mkdir(parents=True, exist_ok=True)

chunks_path = PROJECT_ROOT / "data" / "processed" / "rtve_corpus_chunks.csv"
df_chunks.to_csv(chunks_path, index=False)

ejemplos = pd.concat(
    [buscar(q, k=3, metodo="tfidf", nivel="documento").assign(consulta=q) for q in example_queries],
    ignore_index=True,
)
ejemplos_path = PROJECT_ROOT / "outputs" / "tables" / "caso4_busqueda_ejemplos.csv"
ejemplos.to_csv(ejemplos_path, index=False)

index_path = PROJECT_ROOT / "outputs" / "models" / "caso4_tfidf_index.joblib"
joblib.dump({"vectorizer": tfidf_vectorizer,
             "chunk_matrix": tfidf_chunk_matrix,
             "chunk_ids": df_chunks["chunk_id"].tolist()}, index_path)

for p in (chunks_path, ejemplos_path, index_path):
    print("Guardado:", p.relative_to(PROJECT_ROOT))

Guardado: data/processed/rtve_corpus_chunks.csv
Guardado: outputs/tables/caso4_busqueda_ejemplos.csv
Guardado: outputs/models/caso4_tfidf_index.joblib


## 15. Riesgos y limitaciones

- **Documentos muy largos:** se han fragmentado en *chunks* solapados; el solape mitiga el corte de ideas, pero el tamaño del fragmento es un hiperparámetro que conviene ajustar en la entrega final.
- **Ruido OCR:** afecta tanto al baseline como a los embeddings; el `min_df` de TF-IDF y la lematización futura pueden reducir su impacto.
- **Evaluación sin *ground truth* humano:** la métrica *known-item* valida la consistencia del índice, no la relevancia percibida por un usuario. Una evaluación de relevancia requeriría un conjunto de preguntas anotadas.
- **Solape entre documentos muy parecidos** (p. ej. la serie de vistas orales): puede producir recomendaciones redundantes; una deduplicación o agrupación por similitud lo atenuaría.
- **Capa LLM:** solo es segura si se limita estrictamente a los fragmentos recuperados; el *prompt* del sistema lo impone y obliga a reconocer la falta de evidencia.

En todos los casos el sistema prioriza la **trazabilidad**: cada resultado muestra `doc_id`, título, fragmento de evidencia y enlace al PDF original.

## 16. Conclusiones

El mini caso queda implementado de principio a fin como un **asistente documental semántico** funcional sobre el corpus RTVE 23-F:

- El corpus se fragmenta en *chunks* trazables y se representa con **TF-IDF** (baseline) y, cuando hay conexión, con **embeddings multilingües** (mejora semántica).
- La función `buscar` resuelve la **búsqueda por pregunta** y `recomendar_documentos` ofrece **documentos relacionados**, ambas con similitud coseno y devolviendo evidencias trazables.
- La **evaluación cuantitativa** *known-item* aporta una medida objetiva (recall@k y MRR) del comportamiento del índice, complementando la revisión cualitativa de las consultas de ejemplo.
- La **capa RAG** con LLM queda preparada como extensión opcional y controlada por evidencias, sin convertirse en el punto de partida.

El asistente es, por tanto, una base sólida y reproducible: a nivel de documento recupera con fiabilidad el material correcto y, a nivel de fragmento, localiza el pasaje relevante dentro de documentos extensos, manteniendo siempre el enlace al documento original.